In [136]:
import numpy as np
import pandas as pd
import pymongo
import json
import pprint
import copy
from bson.son import SON

In [228]:
import time, sys
if sys.version_info[0] >= 3 and sys.version_info[1] >= 3:
    timer = time.perf_counter
else:
    timer = time.clock if sys.platform[:3] == 'win' else time.time

# Adaugarea Bazei de date

In [ ]:
#client = pymongo.MongoClient('mongodb+srv://uuu:********************************************************************')
client = pymongo.MongoClient('mongodb+srv://presenter:********************************************************************')
for db in client.list_databases():
    print(db)

{'name': 'bankdb', 'sizeOnDisk': 36986880, 'empty': False}
{'name': 'sample_airbnb', 'sizeOnDisk': 109006848, 'empty': False}
{'name': 'sample_analytics', 'sizeOnDisk': 9715712, 'empty': False}
{'name': 'sample_geospatial', 'sizeOnDisk': 2695168, 'empty': False}
{'name': 'sample_mflix', 'sizeOnDisk': 54812672, 'empty': False}
{'name': 'sample_restaurants', 'sizeOnDisk': 7254016, 'empty': False}
{'name': 'sample_supplies', 'sizeOnDisk': 1204224, 'empty': False}
{'name': 'sample_training', 'sizeOnDisk': 60805120, 'empty': False}
{'name': 'sample_weatherdata', 'sizeOnDisk': 2961408, 'empty': False}
{'name': 'admin', 'sizeOnDisk': 331776, 'empty': False}
{'name': 'local', 'sizeOnDisk': 20666167296, 'empty': False}


In [25]:
db = client.sample_mflix
print(db.list_collection_names())

['sessions', 'comments', 'users', 'movies', 'theaters']


In [287]:
movies = db['movies']
#theaters = db['theaters']
#comments = db['comments']

In [39]:
comments_5 = db.comments.find().limit(5)
for comm in comments_5:
    print(comm)

{'_id': ObjectId('5a9427648b0beebeb6957a42'), 'name': 'Davos Seaworth', 'email': 'liam_cunningham@gameofthron.es', 'movie_id': ObjectId('573a1390f29313caabcd592c'), 'text': 'Illo amet aliquid molestias repellat modi reprehenderit. Nobis totam dicta accusamus voluptates. Eaque distinctio nostrum accusamus eos inventore iste iste sapiente.', 'date': datetime.datetime(2010, 6, 14, 5, 19, 24)}
{'_id': ObjectId('5a9427648b0beebeb6957d0c'), 'name': 'Christopher Robinson', 'email': 'christopher_robinson@fakegmail.com', 'movie_id': ObjectId('573a1391f29313caabcd96ff'), 'text': 'Quam iusto atque illo eaque. Unde repudiandae blanditiis excepturi.', 'date': datetime.datetime(2008, 4, 4, 11, 39, 51)}
{'_id': ObjectId('5a9427648b0beebeb6957ce1'), 'name': 'Anthony Thompson', 'email': 'anthony_thompson@fakegmail.com', 'movie_id': ObjectId('573a1391f29313caabcd95ed'), 'text': 'Debitis qui at sequi nostrum in impedit itaque impedit. Aperiam quibusdam fugit iure fugit voluptatem. Accusamus nihil quidem 

# IPOTEZE 
1. Care este filmul cu cele mai multe recenzii de la critici ?
2. Care sunt numele filmelor ce au mai mult de 20 recenzii de la critici ?
3. Pentru ce tip de filme sunt mai multe recenzii de la spectatori ?

# IMPLEMENTARE

In [108]:
pprint.pprint(movies.find_one())

{'_id': ObjectId('573a1390f29313caabcd548c'),
 'awards': {'nominations': 0, 'text': '2 wins.', 'wins': 2},
 'cast': ['Lillian Gish', 'Mae Marsh', 'Henry B. Walthall', 'Miriam Cooper'],
 'countries': ['USA'],
 'directors': ['D.W. Griffith'],
 'fullplot': 'Two brothers, Phil and Ted Stoneman, visit their friends in '
             'Piedmont, South Carolina: the family Cameron. This friendship is '
             'affected by the Civil War, as the Stonemans and the Camerons '
             'must join up opposite armies. The consequences of the War in '
             'their lives are shown in connection to major historical events, '
             "like the development of the Civil War itself, Lincoln's "
             'assassination, and the birth of the Ku Klux Klan.',
 'genres': ['Drama', 'History', 'Romance'],
 'imdb': {'id': 4972, 'rating': 6.8, 'votes': 15715},
 'lastupdated': '2015-09-11 00:32:27.763000000',
 'plot': "The Civil War divides friends and destroys families, but that's "
       

## Filmul cu cele mai multe recenzii de la critici.

In [140]:
# TEST - sa le impreunez ||| NU MERGE
pipeline =[
    {
        '$group':{'_id':'null', 'max_critic_reviews':{'$max':'$tomatoes.critic.numReviews'}}
    },
    {
        '$match':{'tomatoes.critic.numReviews':'$max_critic_reviews'}
    }
]
result = movies.aggregate(pipeline)
for r in result:
    pprint.pprint(['reviews:', r['max_critic_reviews']])

In [291]:
start = timer()
pipeline =[
    {
        '$group':{'_id':'null', 'max_critic_reviews':{'$max':'$tomatoes.critic.numReviews'}}
    }
]
result = movies.aggregate(pipeline)
print('Runtime: ', timer() - start, '\n')
for r in result:
    pprint.pprint(['Number of Reviews:', r['max_critic_reviews']])

Runtime:  0.41621500000474043 

['Number of Reviews:', 329]


In [ ]:
result = movies.find().max('tomatoes.critic.numReviews')

In [84]:
pipeline = [
    {
        '$match':{'tomatoes.critic.numReviews':329}
    }
]
result = movies.aggregate(pipeline)
for r in result:
    pprint.pprint(['Title:', r['title']])

['Title:', 'Star Trek']


In [293]:
result = db.movies.find({'tomatoes.critic.numReviews':329})
for r in result:
    pprint.pprint(['Title:', r['title']])

['Title:', 'Star Trek']


## Numele si numarul total al filmelor ce au mai mult de 150 recenzii de la critici si viewer rating-ul mai mare si egal cu 3.7.

In [300]:
start = timer()
pipeline = [
    {
        '$match':{
            '$and':[{'tomatoes.critic.numReviews':{'$gte':150}},{'tomatoes.viewer.rating':{'$gte':3.7}}]
        }
    },
    {
        '$sort':{
            'title':1
        }
    }
]
result = movies.aggregate(pipeline)
print('Runtime: ', timer() - start, '\n')
for r in result:
    pprint.pprint(['Title:', r['title']])

Runtime:  17.35080869999365 

['Title:', '(500) Days of Summer']
['Title:', '127 Hours']
['Title:', '13 Hours in a Warehouse']
['Title:', '21 Jump Street']
['Title:', '22 Jump Street']
['Title:', '25th Hour']
['Title:', '300']
['Title:', '50/50']
['Title:', 'A Beautiful Mind']
['Title:', 'A Single Man']
['Title:', 'Across the Universe']
['Title:', 'All Around Us']
['Title:', 'Almost Famous']
['Title:', 'American Beauty']
['Title:', 'American Gangster']
['Title:', 'American Hustle']
['Title:', 'American Sniper']
['Title:', 'American Splendor']
['Title:', 'Amy']
['Title:', 'Amèlie']
['Title:', 'An Education']
['Title:', 'Animal Kingdom']
['Title:', 'Ant-Man']
['Title:', 'Apocalypto']
['Title:', 'Argo']
['Title:', 'Argo']
['Title:', 'Arthur Christmas']
['Title:', 'Atonement']
['Title:', 'Attack the Block']
['Title:', 'Avatar']
['Title:', 'Avengers: Age of Ultron']
['Title:', 'Beasts of the Southern Wild']
['Title:', 'Before Midnight']
['Title:', 'Big Fish']
['Title:', 'Big Hero 6']
['Titl

In [302]:
start = timer()
result = db.movies.find({'tomatoes.critic.numReviews':{'$gte':150},'tomatoes.viewer.rating':{'$gte':3.7}}).sort('title',1)
print('Runtime: ', timer() - start, '\n')
for r in result:
    pprint.pprint(['Title:', r['title']])

Runtime:  0.0003349000180605799 

['Title:', '(500) Days of Summer']
['Title:', '127 Hours']
['Title:', '13 Hours in a Warehouse']
['Title:', '21 Jump Street']
['Title:', '22 Jump Street']
['Title:', '25th Hour']
['Title:', '300']
['Title:', '50/50']
['Title:', 'A Beautiful Mind']
['Title:', 'A Single Man']
['Title:', 'Across the Universe']
['Title:', 'All Around Us']
['Title:', 'Almost Famous']
['Title:', 'American Beauty']
['Title:', 'American Gangster']
['Title:', 'American Hustle']
['Title:', 'American Sniper']
['Title:', 'American Splendor']
['Title:', 'Amy']
['Title:', 'Amèlie']
['Title:', 'An Education']
['Title:', 'Animal Kingdom']
['Title:', 'Ant-Man']
['Title:', 'Apocalypto']
['Title:', 'Argo']
['Title:', 'Argo']
['Title:', 'Arthur Christmas']
['Title:', 'Atonement']
['Title:', 'Attack the Block']
['Title:', 'Avatar']
['Title:', 'Avengers: Age of Ultron']
['Title:', 'Beasts of the Southern Wild']
['Title:', 'Before Midnight']
['Title:', 'Big Fish']
['Title:', 'Big Hero 6']
['

In [252]:
start = timer()
pipeline = [
    {
        '$match':{
            '$and':[{'tomatoes.critic.numReviews':{'$gte':150}},{'tomatoes.viewer.rating':{'$gte':3.7}}]
        }
    },
    {
        '$count':'numMovies'
    }
]
result = movies.aggregate(pipeline)
print('Runtime: ', timer() - start, '\n')
for r in result:
    pprint.pprint(r['numMovies'])

Runtime:  0.06348849998903461 

350


In [303]:
result = movies.find({'tomatoes.critic.numReviews':{'$gte':150},'tomatoes.viewer.rating':{'$gte':3.7}}) #trebuia cu unwind

AttributeError: 'Cursor' object has no attribute 'count'

## Genurile de filme care au mai multe de 5000 de recenzii de la spectatori si o nota mai mare de 3.7

In [250]:
start = timer()
pipeline = [
    {
        '$match':{
            '$and':[{'tomatoes.viewer.numReviews':{'$gte':5000}},{'tomatoes.viewer.rating':{'$gt':3.7}}]
        }
    },
    {
        '$group':{
            '_id':'$genres'
        }
    }
]
result = movies.aggregate(pipeline)
print('Runtime: ', timer() - start, '\n')
for r in result:
    pprint.pprint(r)

Runtime:  0.07421650001197122 

{'_id': ['Action', 'Drama', 'War']}
{'_id': ['Drama', 'Romance', 'Thriller']}
{'_id': ['Drama', 'Comedy', 'Family']}
{'_id': ['Action', 'Crime', 'Drama']}
{'_id': ['Documentary', 'Adventure']}
{'_id': ['Comedy', 'Romance', 'Western']}
{'_id': ['Crime', 'Drama']}
{'_id': ['Drama', 'Fantasy', 'Music']}
{'_id': ['Adventure', 'Mystery', 'Sci-Fi']}
{'_id': ['Action', 'Biography', 'Drama']}
{'_id': ['Animation', 'Music', 'Romance']}
{'_id': ['Documentary', 'History', 'News']}
{'_id': ['Action', 'Adventure', 'Sci-Fi']}
{'_id': ['Action', 'Fantasy', 'War']}
{'_id': ['Drama', 'Mystery', 'Sci-Fi']}
{'_id': ['Drama', 'Horror']}
{'_id': ['Drama', 'History', 'War']}
{'_id': ['Fantasy', 'Horror']}
{'_id': ['Drama', 'History', 'Western']}
{'_id': ['Romance', 'Comedy']}
{'_id': ['Animation', 'Mystery', 'Sci-Fi']}
{'_id': ['Action', 'Comedy', 'History']}
{'_id': ['Crime', 'Drama', 'Mystery']}
{'_id': ['Action', 'Adventure', 'Family']}
{'_id': ['Film-Noir', 'Mystery', 'Th

## Actorul care apare in cele mai multe filme.


In [199]:
pipeline = [
    {
        '$group':{
            '_id':'cast'
        }
    },
    
]
result = movies.aggregate(pipeline)
for r in result:
    pprint.pprint(r)

{'_id': 'cast'}


## Titlul filmelor si actorii care apar impreuna cu actorul "Johnny Deep".

In [285]:
start = timer()
match_state = {
        '$match':{
            '$and':[
                {'cast':{'$in':['Johnny Depp',['$cast']]}},
                {'year':{'$gte':2000}},
                {'imdb.rating':{'$gte':7.5}}
            ]
        }
    }
limit_state = { '$limit':5}
group_state = {
    '$group':{
        '_id':'$title'
    }
}
pipeline = [match_state,limit_state,group_state]
result = movies.aggregate(pipeline)
print('Runtime: ', timer() - start, '\n')
for r in result:
    #pprint.pprint(['Title:', r['title'], 'Actors:', r['cast']])
    pprint.pprint(r)

Runtime:  1.5704229999973904 

{'_id': 'Finding Neverland'}
{'_id': 'Pirates of the Caribbean: The Curse of the Black Pearl'}
{'_id': 'Sweeney Todd: The Demon Barber of Fleet Street'}
{'_id': 'Blow'}


In [284]:
start = timer()
result = movies.find({'cast':{'$in':['Johnny Depp',['$cast']]},'year':{'$gte':2000},'imdb.rating':{'$gte':7.5}})
print('Runtime: ', timer() - start, '\n')
for r in result:
    pprint.pprint(r)

Runtime:  0.0002652999828569591 

{'_id': ObjectId('573a13a1f29313caabd086cd'),
 'awards': {'nominations': 5, 'text': '1 win & 5 nominations.', 'wins': 1},
 'cast': ['Johnny Depp', 'Penèlope Cruz', 'Franka Potente', 'Rachel Griffiths'],
 'countries': ['USA'],
 'directors': ['Ted Demme'],
 'fullplot': 'A boy named George Jung grows up in a struggling family in the '
             "1950's. His mother nags at her husband as he is trying to make a "
             "living for the family. It is finally revealed that George's "
             'father cannot make a living and the family goes bankrupt. George '
             'does not want the same thing to happen to him, and his friend '
             "Tuna, in the 1960's, suggests that he deal marijuana. He is a "
             "big hit in California in the 1960's, yet he goes to jail, where "
             'he finds out about the wonders of cocaine. As a result, when '
             'released, he gets rich by bringing cocaine to America. However, '
 

## Numarul filmelor de razboi ce au aparut dupa 1970 si au o nota imdb mai mare de 5.5

In [230]:
start = timer()
match_state = {
    '$match':{
        '$and':[
            {'genres':{'$in':['War',['$genres']]}},
            {'year':{'$gt':1970}},
            {'imdb.rating':{'$gt':5.5}}
        ]
    }
}
count_state = {
    '$count':'num_of_movies'
}
pipeline = [
    match_state,
    count_state
]
result = movies.aggregate(pipeline)
print('Runtime: ', timer() - start, '\n')
for r in result:
    pprint.pprint(r['num_of_movies'])

Runtime:  0.08053470001323149 

507


## Nota medie a filmelor de gen "Romance" care au aparut inainte de anii 1990

In [229]:
start = timer()
match_state = {
    '$match':{
        '$and':[
            {'genres':{'$in':['Romance',['$genres']]}},
            {'year':{'$lt':1990}}
        ]
    }
}
group_state = {
    '$group':{
        '_id':'null',
        'averageRating':{'$avg':'$imdb.rating'}
    }
}
pipeline = [
    match_state,
    group_state
]
result = movies.aggregate(pipeline)
print('Runtime: ', timer() - start, '\n')
for r in result:
    pprint.pprint(['Media Rating-ului:', r['averageRating']])

Runtime:  0.15685200001462363 

['Media Rating-ului:', 7.0609271523178805]
